In [1]:
import pandas as pd
RAW_PATH = "SuperStore Sales DataSet.xlsx"
df = pd.read_excel(RAW_PATH)
print(f"Raw shape: {df.shape}")

Raw shape: (5901, 23)


In [2]:
df

,Row ID+O6G3A1:R6,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Category,Sub-Category,Product Name,Sales,Quantity,Profit,Returns,Payment Mode,ind1,ind2
0,4918,CA-2019-160304,2019-01-01,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",73.940,1,28.2668,NaN,Online,NaN,NaN
1,4919,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",173.940,3,38.2668,NaN,Online,NaN,NaN
2,4920,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Technology,Phones,GE 30522EE2,231.980,2,67.2742,NaN,Cards,NaN,NaN
3,3074,CA-2019-125206,2019-01-03,2019-01-05,First Class,LR-16915,Lena Radford,Consumer,United States,Los Angeles,...,Office Supplies,Storage,Recycled Steel Personal File for Hanging File ...,114.460,2,28.6150,NaN,Online,NaN,NaN
4,8604,US-2019-116365,2019-01-03,2019-01-08,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,Technology,Accessories,Imation Clip USB flash drive - 8 GB,30.080,2,-5.2640,NaN,Online,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5896,907,CA-2020-143259,2020-12-30,2021-01-03,Standard Class,PO-18865,Patrick O'Donnell,Consumer,United States,New York City,...,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Fully Ass...",213.136,4,12.1176,NaN,COD,NaN,NaN
5897,1297,CA-2020-115427,2020-12-30,2021-01-03,Standard Class,EB-13975,Erica Bern,Corporate,United States,Fairfield,...,Office Supplies,Binders,"Cardinal Slant-D Ring Binder, Heavy Gauge Vinyl",295.904,2,4.5188,1.0,Online,NaN,NaN
5898,5092,CA-2020-156720,2020-12-30,2021-01-03,Standard Class,JM-15580,Jill Matthias,Consumer,United States,Loveland,...,Office Supplies,Fasteners,Bagged Rubber Bands,388.024,3,-0.6048,NaN,Online,NaN,NaN
5899,909,CA-2020-143259,2020-12-30,2021-01-03,Standard Class,PO-18865,Patrick O'Donnell,Consumer,United States,New York City,...,Office Supplies,Binders,Wilson Jones Legal Size Ring Binders,462.776,3,19.7910,NaN,COD,NaN,NaN


In [3]:
df = df.drop(columns=["ind1", "ind2"])
df = df.rename(columns={"Row ID+O6G3A1:R6": "Row ID"})
df["Returns"] = df["Returns"].fillna(0)
df["Is Returned"] = df["Returns"].astype(int).astype(bool)
df = df.drop(columns=["Returns"])

In [5]:
dupes = df.duplicated().sum()
if dupes:
    df = df.drop_duplicates()
print(f"Number of duplicate rows removed: {dupes}")

Number of duplicate rows removed: 0


In [6]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

In [8]:
df["Order Processing Time"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Margin %"] = (df["Profit"] / df["Sales"]).round(4) * 100
first_purchase = df.groupby("Customer ID")["Order Date"].transform("min")
df["Customer First Purchase Date"] = first_purchase
order_counts = df.groupby("Customer ID")["Order ID"].transform("nunique")
df["Is Repeat Customer"] = order_counts > 1

In [9]:
print(f"\nFinal shape: {df.shape}")
print(f"Nulls remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nColumns: {df.columns.tolist()}")


Final shape: (5901, 25)
Nulls remaining:
Series([], dtype: int64)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Profit', 'Payment Mode', 'Is Returned', 'Order Processing Time', 'Profit Margin %', 'Customer First Purchase Date', 'Is Repeat Customer']


In [10]:
import numpy as np
np.random.seed(42)
DISCOUNT = [0.00, 0.10, 0.20, 0.30]
df["Discount"] = np.random.choice(DISCOUNT, size=len(df))

In [11]:
df.to_excel("superstore_cleaned.xlsx", index=False)
print("\nSaved: superstore_cleaned.xlsx")


Saved: superstore_cleaned.xlsx
